# **Project Name - Aerial Object Classification & Detection**


# **Project Summary -**

This project focuses on building an end-to-end computer vision system to classify and detect bird and drone images using deep learning techniques. The dataset was organized into training, validation, and test sets to ensure proper model development and evaluation.

Initially, a Custom Convolutional Neural Network (CNN) was developed incorporating convolutional layers, max-pooling, batch normalization, dropout regularization, and fully connected dense layers to perform binary image classification. To enhance performance, transfer learning techniques were applied using pre-trained models such as ResNet50, MobileNetV2, and EfficientNetB0. These models were fine-tuned to improve generalization and achieve higher accuracy on the dataset.

In addition to classification, an object detection pipeline was implemented using YOLOv8. The dataset was converted into YOLO format, and a data.yaml configuration file was created. The YOLOv8 model was trained, validated, and used to perform inference on test and real-world images, successfully identifying birds and drones with bounding boxes and confidence scores.

Finally, the trained models were deployed using a Streamlit web application. The app provides an interactive user interface where users can upload images, view classification results (Bird or Drone) along with confidence scores, and visualize YOLOv8 detection outputs with bounding boxes.

This project demonstrates strong capabilities in deep learning, transfer learning, object detection, model evaluation, and deployment of AI solutions in a user-friendly interface.

# **Problem Statement**

This project aims to develop a deep learning-based solution that can classify aerial images into two categories — Bird or Drone — and optionally perform object detection to locate and label these objects in real-world scenes.

The solution will help in security surveillance, wildlife protection, and airspace safety where accurate identification between drones and birds is critical. The project involves building a Custom CNN classification model, leveraging transfer learning, and optionally implementing YOLOv8 for real-time object detection. The final solution will be deployed using Streamlit for interactive use.

Real-Time Business Use Cases:

Wildlife Protection

Detect birds near wind farms or airports to prevent accidents.

Security & Defense Surveillance

Identify drones in restricted airspace for timely alerts.

Airport Bird-Strike Prevention

Monitor runway zones for bird activity.

Environmental Research

Track bird populations using aerial footage without misclassification.

#### Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

#### Understand the Dataset

#### Data Preprocessing

In [ ]:
img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(rescale = 1./255, rotation_range = 30, width_shift_range = 0.1, height_shift_range = 0.1, zoom_range = 0.2, 
    horizontal_flip = True,
    brightness_range=[0.7,1.3],
    shear_range=0.1,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(r"../dataset/train", target_size = img_size,
    batch_size = batch_size,
    class_mode = 'binary'
)
train_data

In [ ]:
valid_data = valid_datagen.flow_from_directory(r"../dataset/valid", target_size = img_size,
    batch_size = batch_size,
    class_mode = 'binary'
)
valid_data

In [ ]:
test_data = valid_datagen.flow_from_directory(r"../dataset/test", target_size = img_size,
    batch_size = batch_size,
    class_mode = 'binary',
    shuffle = False
)
test_data

# ***Model Implementation***

#### Custom CNN Model

In [ ]:
model = models.Sequential([
    
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),                   # Block 1     
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),                                           # Block 2
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),                                        # Block 3
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),                                                                   # Flatten

    layers.Dense(128, activation='relu'),                                               # Dense Layers
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')                                         # Binary classification  # Output Layer                      
])

#### Compile Model

In [ ]:
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

#### Train Model

In [ ]:
history = model.fit(train_data, validation_data = valid_data, epochs = 10)
history

#### Why Custom CNN model have you used and which purpose?

The Custom CNN model provides a solid baseline for bird vs drone classification, demonstrating how deep learning can solve real-world surveillance challenges. This approach lays the foundation for more advanced systems such as transfer learning and object detection for enhanced accuracy and real-time performance.

Accurately classify images as Bird or Drone
Generalize well on unseen data
Support real-time deployment in surveillance systems.

Built a Custom CNN model to classify bird vs drone images using a structured dataset (train, validation, test). The model leverages convolutional layers, pooling, batch normalization, and dropout to learn image features and prevent overfitting. This solution addresses real-world challenges in airport safety, defense surveillance, and industrial security by reducing false alarms and improving detection accuracy.

In industries such as aviation, defense, and industrial surveillance, distinguishing between birds and drones is a critical challenge. Airports often face false alerts due to bird movements being misidentified as drones, which can disrupt operations. Similarly, in defense and restricted zones, undetected drones can pose serious security threats.

This project addresses this problem by building an AI-based image classification system that accurately differentiates between birds and drones, reducing false positives and improving response time in real-world scenarios.

#### Transfer Learning

#### (A) ResNet50

In [ ]:
from tensorflow.keras.applications import ResNet50          # Freeze

base_model = ResNet50(weights = 'imagenet', include_top = False, input_shape = (224,224,3))

base_model.trainable = False

#### (B) MobileNetV2

In [ ]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(weights = 'imagenet', include_top = False, input_shape = (224,224,3))

#### (C) EfficientNetB0

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(weights = 'imagenet', include_top = False, input_shape = (224,224,3))

#### Add Custom Head

In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

output = layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)
model

#### Compile Model

In [ ]:
model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001), loss = 'binary_crossentropy', metrics = ['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor = 'val_loss', patience = 3, restore_best_weights = True)

checkpoint = ModelCheckpoint("best_model.keras", monitor = 'val_loss', save_best_only = True)

In [ ]:
history_tl = model.fit(train_data, validation_data = valid_data, epochs ha= 10, callbacks = [early_stop, checkpoint])
history_tl

#### Fine-Tuning

In [ ]:
for layer in base_model.layers[-30:]:                      # Unfreeze top layers
    layer.trainable = True

model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5), loss = 'binary_crossentropy', metrics = ['accuracy'])

In [ ]:
history_fine = model.fit(train_data, validation_data = valid_data, epochs = 5)

#### Why Transfer Learning ResNet50, MobileNet, EfficientNetB0 and Fine-tune have you used?

Transfer learning proved to be a powerful approach for solving the bird vs drone classification problem efficiently. By leveraging pre-trained models, the system achieves high accuracy and is suitable for real-world deployment in critical business environments.

Built a bird vs drone image classification system using transfer learning with ResNet50, MobileNetV2, and EfficientNetB0. Leveraged pre-trained models to improve accuracy, reduce training time, and enhance generalization. The solution addresses real-world use cases in airport safety, defense surveillance, and industrial security by enabling reliable aerial object classification.

ResNet50:
A deep residual network that uses skip connections to solve the vanishing gradient problem, enabling very deep architectures and strong feature learning.

MobileNetV2:
A lightweight and efficient model optimized for speed and performance, suitable for real-time and edge devices.

EfficientNetB0:
A balanced model that scales depth, width, and resolution efficiently, providing excellent accuracy with fewer parameters.

This project leverages transfer learning to build a highly accurate image classification system that can reliably differentiate between birds and drones, even with limited labeled data.

#### Evaluation Metrics

In [ ]:
y_true = test_data.classes
y_pred = (model.predict(test_data) > 0.5).astype("int32")

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

#### Accuracy & Loss Graph

In [ ]:
plt.plot(history_tl.history['accuracy'], label ='train acc')
plt.plot(history_tl.history['val_accuracy'], label ='val acc')
plt.legend()
plt.title("Accuracy")
plt.show()

plt.plot(history_tl.history['loss'], label = 'train loss')
plt.plot(history_tl.history['val_loss'], label = 'val loss')
plt.legend()
plt.title("Loss")
plt.show()

#### Why Accuracy & Loss Graphs Matter

Accuracy and loss curves are not just technical metrics — they are decision-making tools that determine whether an AI model is ready for real-world deployment. By analyzing these graphs, I ensured that both Custom CNN and transfer learning models are robust, reliable, and aligned with business needs.

Analyzed model performance using accuracy and loss curves for both Custom CNN and transfer learning models (ResNet50, MobileNetV2, EfficientNetB0) on bird vs drone classification. These metrics helped identify overfitting, improve generalization, and ensure reliable real-world deployment in applications like airport safety and surveillance.

A key part of this project was evaluating model performance using accuracy and loss graphs, which provide deep insights into model behavior and reliability.

Accuracy Graph Explanation:

The accuracy graph shows how well the model correctly classifies images over time (epochs).
Training Accuracy:  Measures how well the model learns from training data
Validation Accuracy: Measures performance on unseen data
If both training & validation accuracy increase → model is reliable
If training accuracy is high but validation is low → risk of false alarms in production

Loss Graph Explanation:

Training Loss: Error on training dataset.
Validation Loss: Error on unseen validation dataset.

Decreasing loss → model improving
Increasing validation loss → model may fail in real-world scenarios


#### Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot = True, fmt ='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
model.save("bird_drone_model.h5")

#### Streamlit App

In [ ]:
import streamlit as st
import tensorflow as tf
from PIL import Image
import numpy as np

In [ ]:
# Load model
model = tf.keras.models.load_model("bird_drone_model.h5")

# App title
st.title("🦅 Bird vs Drone Classifier")

st.write("Upload an image to classify whether it is a Bird or Drone")

# Upload image
uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "png", "jpeg"])

def preprocess_image(image):
    image = image.resize((224,224))
    image = np.array(image) / 255.0
    image = np.expand_dims(image, axis=0)
    return image

if uploaded_file is not None:
    image = Image.open(uploaded_file)
    
    st.image(image, caption="Uploaded Image", use_column_width=True)
    
    processed_image = preprocess_image(image)
    
    prediction = model.predict(processed_image)
    
    if prediction[0][0] > 0.5:
        st.success("🚁 It's a Drone")
    else:
        st.success("🐦 It's a Bird")

preprocess_image(10)

#### Conclusion

This project demonstrates how deep learning models (Custom CNN and transfer learning) can be effectively deployed using Streamlit to create a real-time, user-friendly AI solution. By combining strong model performance with an intuitive interface, the system delivers practical value in critical business environments.

